# 02 - Raw schema inspection and DuckLake ingest

Gate 4: verify the DuckLake raw layer end to end (create, reopen, reconcile, re-run).

Expected raw-layer facts, from `reports/schema_quality.md`:

- 903,653 session rows
- 902,755 distinct `(fullVisitorId, visitId)`
- 11,552 sessions with `totals.transactions`
- dates 20160801 -> 20170801

Reusable logic lives in `src/analytics_ml`; this notebook only orchestrates and displays it.
`warehouse/` is Git-ignored, so the catalog and Parquet files stay local.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

from analytics_ml.ingestion import inspect_raw
from analytics_ml.lakehouse import ducklake_setup as dl

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

print("duckdb:", duckdb.__version__)
print("catalog:", dl.CATALOG_PATH.relative_to(dl.PROJECT_ROOT).as_posix())
print("raw glob:", inspect_raw.RAW_GLOB_LABEL)

## 1. Raw Parquet inspection

In [ ]:
raw = inspect_raw.inspect_raw()
overall = raw["overall"]

pd.DataFrame(
    {
        "metric": [
            "sessions",
            "distinct_sessions",
            "transaction_sessions",
            "first_day",
            "last_day",
        ],
        "raw_parquet": [
            int(overall["sessions"]),
            int(overall["distinct_sessions"]),
            int(overall["transaction_sessions"]),
            overall["first_day"],
            overall["last_day"],
        ],
    }
)

In [ ]:
raw["quarter_rows"][
    ["quarter", "sessions", "distinct_sessions", "transaction_sessions", "first_day", "last_day"]
]

In [ ]:
pd.DataFrame(raw["null_rows"], columns=["field", "null_rows"])

## 2. DuckLake ingest and reopen

`ensure_raw_table` is idempotent: it creates `raw.ga_sessions` only when it is missing,
then checkpoints. Re-running it must not append rows or create a snapshot.
`run_verification` also reopens the catalog through a brand-new read-only connection.

In [ ]:
result = dl.run_verification()
{
    "created_now": result["created_now"],
    "created_again_on_rerun": result["created_again"],
}

### Runtime versions

In [ ]:
pd.DataFrame(result["versions"].items(), columns=["component", "value"])

### Reconciliation: raw Parquet vs DuckLake

In [ ]:
pd.DataFrame(result["reconciliation"])

In [ ]:
pd.DataFrame(result["quarters"])[["quarter", "sessions", "raw_sessions", "match"]]

### Persistence across a new connection

In [ ]:
reopened = result["reopened"]
{
    "read_only_connection_rows": reopened["sessions"],
    "snapshot_id": reopened["snapshot_id"],
    "matches_lake": reopened["sessions"] == result["lake"]["sessions"],
}

### Idempotency

In [ ]:
result["idempotency"]

## 3. Verdict and evidence report

In [ ]:
print("reconciliation:", all(row["match"] for row in result["reconciliation"]))
print("quarters:", result["quarters_match"])
print("idempotent:", result["idempotency"]["ok"])
print("matches expected facts:", result["matches_expected"])
print("overall:", "PASS" if result["ok"] else "FAIL")

report = dl.write_verification_report(result)
print("wrote:", report.relative_to(dl.PROJECT_ROOT).as_posix())

In [ ]:
assert result["ok"], (
    "DuckLake verification failed - see reports/ducklake_verification.md"
)

## 4. Observed raw schema (nested structures preserved)

In [ ]:
raw["schema"]